# Customer Segmentation using RFM Analysis & K-Means Clustering

Welcome to the **Customer Segmentation** educational notebook! 

In this notebook, we will explore how to segment customers based on their historical buying behavior. Customer segmentation is a powerful tool in modern database marketing, retail analytics, and business intelligence. By grouping customers with similar buying patterns, businesses can tailor their product offerings, marketing campaigns, and customer support strategies to maximize Customer Lifetime Value (LTV) and reduce churn.

## What is RFM Analysis?
RFM stands for:
- **Recency (R)**: How recently did a customer make a purchase? (Lower is usually better, indicating they are still active).
- **Frequency (F)**: How often does the customer buy? (Higher is better, indicating loyalty).
- **Monetary Value (M)**: How much total revenue does the customer generate? (Higher is better, indicating value).

## Steps Covered in this Notebook:
1. **Exploratory Data Analysis (EDA)** & importing libraries.
2. **Feature Engineering**: Transforming raw transactional data into customer-level RFM metrics.
3. **Feature Preprocessing**: Scaling and normalizing the data.
4. **Hyperparameter Tuning**: Finding the optimal number of clusters ($k$) using the **Elbow Method** and **Silhouette Coefficient**.
5. **Model Training**: Fitting the **K-Means** clustering algorithm using `scikit-learn`.
6. **Visualizing Clusters**: Plotting Recency, Frequency, and Monetary distributions in 2D and 3D.
7. **Persona Profiling**: Reviewing stats of each group and defining actionable targeting recommendations.

---
## Step 1: Import Libraries & Load Data
First, we will load standard python packages including `pandas`, `numpy`, `scikit-learn`, `matplotlib`, and `seaborn`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set plotting themes
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("[+] Data science libraries loaded successfully!")

### Load Sales Transactional Data
We will check if there is an exported CSV from the RevIQ web dashboard (e.g. `sales_export.csv`). If you don't have one, make sure to export one from your dashboard page or run the script.

In [ ]:
csv_path = "sales-dashboard/sales_export_sample.csv"
# Let's check a few paths or list directories if needed
if not os.path.exists(csv_path):
    # Fallback to local workspace files
    csv_path = "sales_export.csv" 

print(f"[*] Looking for CSV at: {csv_path}")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"[+] Successfully loaded dataset containing {df.shape[0]} rows and {df.shape[1]} columns.")
    display(df.head())
else:
    print("[-] CSV file not found yet. Please run Export CSV from the RevIQ dashboard first!")

---
## Step 2: Aggregate to RFM Metrics
Transactional data has multiple rows per customer. K-Means requires exactly one row per customer. We aggregate: 
- `Recency` = Days since the customer's last order to the day after the last overall order in the dataset.
- `Frequency` = Count of total orders.
- `Monetary` = Sum of Revenue.

In [ ]:
if 'df' in locals():
    # Clean data types
    df['Date'] = pd.to_datetime(df['Date'])
    df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce').fillna(0)
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').fillna(1)
    
    # Calculate Reference Date
    ref_date = df['Date'].max() + pd.Timedelta(days=1)
    print(f"[*] Reference Date for Recency calculation: {ref_date.strftime('%Y-%m-%d')}")
    
    # Group by CustomerID
    rfm = df.groupby('CustomerID').agg({
        'Date': lambda x: (ref_date - x.max()).days,
        'Sales': 'sum',
        'Revenue': 'sum',
        'CustomerName': 'first'
    }).reset_index()
    
    rfm.rename(columns={
        'Date': 'Recency',
        'Sales': 'Frequency',
        'Revenue': 'Monetary'
    }, inplace=True)
    
    # Remove zero/negative revenue outliers
    rfm = rfm[rfm['Monetary'] > 0]
    
    print(f"[+] Engineered RFM profiles for {rfm.shape[0]} unique customers!")
    display(rfm.head())
else:
    print("[-] Dataframe 'df' is not loaded yet.")

---
## Step 3: Feature Scaling (Normalization)
K-Means clustering uses **Euclidean Distance** to measure similarity. Since Recency (days), Frequency (counts), and Monetary (dollars) have completely different scales, the Monetary value (usually in thousands) would dominate the distance calculation. We use `StandardScaler` (z-score scaling) to normalize features to have a mean of 0 and standard deviation of 1.

In [ ]:
if 'rfm' in locals():
    features = rfm[['Recency', 'Frequency', 'Monetary']].values
    
    # Scale features
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    
    # Create a scaled dataframe for verification
    df_scaled = pd.DataFrame(scaled_features, columns=['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled'])
    print("[+] Standard Scaling complete. Verification statistics:")
    display(df_scaled.describe())
else:
    print("[-] RFM dataframe is not loaded yet.")

---
## Step 4: Hyperparameter Tuning ($k$ selection)
How do we know how many clusters are correct?
1. **The Elbow Method**: Plots the Within-Cluster Sum of Squares (WCSS/inertia) against $k$. We look for the "elbow" point where the curve starts to bend and level off.
2. **Silhouette Coefficient**: Measures how similar a data point is to its own cluster compared to other clusters. Scores closer to $+1$ indicate excellent separation, while scores close to $-1$ or $0$ represent poor clustering.

In [ ]:
if 'scaled_features' in locals():
    wcss = []
    silhouette_avg = []
    k_range = range(2, 9)
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(scaled_features)
        wcss.append(kmeans.inertia_)
        silhouette_avg.append(silhouette_score(scaled_features, kmeans.labels_))
        
    # Plot the curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    ax1.plot(k_range, wcss, 'o-', color='#4f8ef7', linewidth=2.5, markersize=8)
    ax1.set_title('Elbow Method (Inertia Curve)', fontsize=14)
    ax1.set_xlabel('Number of Clusters (k)')
    ax1.set_ylabel('Inertia (WCSS)')
    ax1.set_xticks(k_range)
    
    ax2.bar(k_range, silhouette_avg, color='#00d4aa', alpha=0.8, edgecolor='#00ab8a')
    ax2.set_title('Silhouette Analysis', fontsize=14)
    ax2.set_xlabel('Number of Clusters (k)')
    ax2.set_ylabel('Average Silhouette Coefficient')
    ax2.set_xticks(k_range)
    
    plt.show()
else:
    print("[-] Scaled features not available.")

---
## Step 5: Fit K-Means Clustering Model
Now we train our finalized K-Means model on the chosen optimal number of clusters (typically $k=4$ represents the perfect balance for customer segmentation, producing distinct segments).

In [ ]:
if 'scaled_features' in locals():
    k_optimal = 4
    print(f"[*] Training finalized K-Means model with k = {k_optimal}...")
    
    kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
    rfm['Cluster'] = kmeans.fit_predict(scaled_features)
    
    print("[+] Training complete. Assigned clusters to individual customers.")
    display(rfm.head())
else:
    print("[-] Scaled features not available.")

---
## Step 6: Visualizing the Clusters
Let's see our mathematical customer groups in standard 2D feature plots and a 3D scatter plot!

In [ ]:
if 'rfm' in locals():
    # Recency vs Frequency
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=rfm, x='Recency', y='Frequency', hue='Cluster', palette='Set1',
        size='Monetary', sizes=(50, 400), alpha=0.75, edgecolor='k'
    )
    plt.title('2D Cluster Visualization: Recency vs Frequency', fontsize=14)
    plt.xlabel('Recency (Days since last order)')
    plt.ylabel('Frequency (Total purchases)')
    plt.legend(title='Cluster Group')
    plt.show()
else:
    print("[-] RFM dataframe not available.")

In [ ]:
if 'rfm' in locals():
    # 3D Plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    colors = plt.cm.Set1(np.linspace(0, 1, k_optimal))
    
    for c in range(k_optimal):
        sub = rfm[rfm['Cluster'] == c]
        ax.scatter(
            sub['Recency'], sub['Frequency'], sub['Monetary'],
            label=f'Cluster {c}', s=80, alpha=0.8, edgecolor='k'
        )
        
    ax.set_title('3D Multi-Dimensional RFM Customer Clusters', fontsize=14)
    ax.set_xlabel('Recency (Days)')
    ax.set_ylabel('Frequency (Purchases)')
    ax.set_zlabel('Monetary Spend ($)')
    ax.legend()
    plt.show()
else:
    print("[-] RFM dataframe not available.")

---
## Step 7: Profiling the Clusters & Targeted Marketing Actions
We can calculate the average RFM scores for each cluster, understand who they are in plain business terms, and map them to targeted marketing recommendations!

In [ ]:
if 'rfm' in locals():
    # Calculate cluster statistics
    cluster_stats = rfm.groupby('Cluster').agg({
        'Recency': 'mean',
        'Frequency': 'mean',
        'Monetary': 'mean',
        'CustomerID': 'count'
    }).rename(columns={'CustomerID': 'Customer Count'}).reset_index()
    
    # Display statistics table
    print("=== Cluster Summary Statistics ===")
    display(cluster_stats)
    
    print("\n=== Persona Maps & Targeted Campaign Strategies ===")
    # Write descriptive summaries
    for _, row in cluster_stats.iterrows():
        idx = int(row['Cluster'])
        count = int(row['Customer Count'])
        r, f, m = row['Recency'], row['Frequency'], row['Monetary']
        
        print(f"\nCluster #{idx} (Size: {count} customers, {count/len(rfm)*100:.1f}%):")
        print(f"  - Stats: Avg Recency={r:.1f} days, Avg Frequency={f:.1f} orders, Avg Monetary Spend=${m:.2f}")
        
        # Determine Persona
        if r < rfm['Recency'].mean() and f >= rfm['Frequency'].mean() and m >= rfm['Monetary'].mean():
            print("  -> Persona: VIP Champions (Highly Active & High Spending)")
            print("  -> Strategy: Reward loyalty with exclusive sneak peeks, early access, and a premium loyalty program.")
        elif r > rfm['Recency'].mean() and f >= rfm['Frequency'].mean() and m >= rfm['Monetary'].mean():
            print("  -> Persona: At-Risk VIPs (Loyal Spenders Who Went Quiet)")
            print("  -> Strategy: Re-engage immediately! Send a high-value incentive discount or personal customer support call.")
        elif r < rfm['Recency'].mean() and f < rfm['Frequency'].mean():
            print("  -> Persona: New / Promising Customers (Recent Buyers with Low Frequency)")
            print("  -> Strategy: Send a welcoming onboarding flow, product guide booklets, and a minor discount for order #2.")
        else:
            print("  -> Persona: Lost / Inactive Customers (Quiet, Low-Spending)")
            print("  -> Strategy: Focus marketing budget elsewhere. Automated, low-cost re-activation email triggers are best.")
else:
    print("[-] RFM dataframe not available.")